# Finals Practice and Training

In [1]:
import pandas as pd, numpy as np

Student_ID = 5212
np.random.seed(Student_ID)
N = 30000

def generateData(n):
    ids = np.arange(1, n+1)
    groups = np.random.choice(["A","B","C"], size=n, p=[0.4,0.35,0.25])
    size = np.random.randint(20, 500, size=n)
    age = np.random.randint(0, 50, size=n)
    rate1 = np.round(np.random.normal(0.5, 0.15, size=n), 4)
    rate2 = np.round(np.random.normal(1.0, 0.4, size=n), 4)
    mask = np.random.rand(n) < 0.03
    rate2[mask] = np.nan
    return pd.DataFrame({"id": ids, "group": groups, "size": size,
                         "age": age, "rate1": rate1, "rate2": rate2})

data = generateData(N)
data.head()

,id,group,size,age,rate1,rate2
0,1,A,365,19,0.4527,0.7316
1,2,B,145,22,0.4540,0.8526
2,3,B,263,49,0.2229,0.9196
3,4,B,223,42,0.3286,1.5200
4,5,B,385,2,0.5340,1.2862


## Task 1

### Step 1
 Write a function `my_mean(values)` that returns the mean of a list without using `sum()`, `len()`, `np.mean()` or `.mean()`.

In [2]:
def my_mean(values):
    total = 0
    count = 0
    for v in values:
        if v== v:
            total += v
            count += 1
    return total/count

In [3]:
for col in ["size","age","rate1","rate2"]:
    values = list(data[col])
    print(col, my_mean(values))

size 260.55856666666665
age 24.534966666666666
rate1 0.5004729000000017
rate2 1.0012718532009195


### Step 2: Chunks

In [10]:
def make_chunks(L,k):
    size = len(L) // k + 1
    return [ L[i:i+size] for i in range(0,len(L),size)]
print(make_chunks([2,4,6,8,10],3))

[[2, 4], [6, 8], [10]]


Write `partial(values)`. It's your `my_mean` with one change: at the end, instead of returning `total / count`, it returns both: `return total, count`.

In [16]:
def partial(values):
    total = 0
    count = 0
    for v in values:
        if v== v:
            total += v
            count += 1
    return total, count

In [17]:
print(partial([2, 4]))

age = list(data["age"])
chunks = make_chunks(age, 4)
parts = [partial(c) for c in chunks] 
print(parts)

(6, 2)
[(185472, 7501), (183214, 7501), (183424, 7501), (183939, 7497)]


In [22]:
big_total = 0
big_count = 0
for t,c in parts:
    big_total += t
    big_count += c
mean = big_total/big_count
print(mean)

24.534966666666666


### Step 4

In [25]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=4) as ex:     # hire 4 workers
    parts = list(ex.map(partial, chunks))          # each worker runs partial on one piece

big_total = 0
big_count = 0
for t,c in parts:
    big_total += t
    big_count += c
mean = big_total/big_count
print(mean)

24.534966666666666


### Step 5

In [30]:
import time
from concurrent.futures import ThreadPoolExecutor

def make_chunks(L,k):
    size = len(L) // k+1
    return [L[i:i+size] for i in range(0,len(L),size)]

# 1. serial version
start = time.perf_counter()
m1 = my_mean(age)
t_serial = time.perf_counter() - start

# 2. thread version (chunks + 4 threads + combine)
start = time.perf_counter()
chunks = make_chunks(age,4)

with ThreadPoolExecutor(max_workers=4) as ex:     # hire 4 workers
    parts = list(ex.map(partial, chunks))          # each worker runs partial on one piece

big_total = 0
big_count = 0
for t,c in parts:
    big_total += t
    big_count += c
m2 = big_total/big_count
t_threads = time.perf_counter() - start

print("Same result?", m1==m2)
print("serial :", t_serial)
print("threads:", t_threads)
print("speed-up =", t_serial / t_threads)

Same result? True
serial : 0.010160199999518227
threads: 0.01706719999856432
speed-up = 0.5953056154713658


## Full answer

In [40]:
#the tools:functions
import time
from concurrent.futures import ThreadPoolExecutor

columns = ["size","age","rate1","rate2"]

def my_mean(values):
    total = 0
    count = o
    for v in values:
        if v == v:
            total += v
            count += 1
        return total/count

def partial(values):
    total = 0
    count = 0
    for v in values:
        if v == v:
            total += v
            count += 1
        return total, count

def make_chunks(L,k):
    size = len(L) // k+1
    return [L[i:i+size] for i in range(0,len(L),size)]

def threaded_mean(values,k):
    chunks = make_chunks(values,k)
    with ThreadPoolExecutor(max_workers=4) as ex:
        parts = list(ex.maps(partial,chunks))
    big_total = 0
    big_count = 0
    for t,c in parts:
        big_total+=t
        big_count+=c
    return big_total/count

In [41]:
# prepare the 4 columns as plain lists (outside the timer)
all_values = []
for col in columns:
    all_values.append(list(data[col]))

# 1. serial
start = time.perf_counter()
serial_means = []
for values in all_values:
    serial_means.append(my_mean(values))
t_serial = time.perf_counter() - start

# 2. threads
start = time.perf_counter()
thread_means = []
for values in all_values:
    thread_means.append(threaded_mean(values, 4))
t_threads = time.perf_counter() - start

# 3. results
for i in range(len(columns)):
    same = round(serial_means[i], 6) == round(thread_means[i], 6)
    print(columns[i], serial_means[i], thread_means[i], same)
print("Serial time :", t_serial)
print("Thread time :", t_threads)
print("Speed-up    :", t_serial / t_threads)

NameError: name 'o' is not defined

### Practice

**Ex1:** Use your thread version to compute the `mean` of `size`, `age`, `rate1`, `rate2` in a loop:

In [37]:
for col in ["size", "age", "rate1", "rate2"]:
    values = list(data[col])
    chunks = make_chunks(values,4)

    with ThreadPoolExecutor(max_workers=4) as ex:
        parts = list(ex.map(partial, chunks))

    big_total = 0 
    big_count = 0
    for t,c in parts:
        big_total += t
        big_count += c
        mean=big_total/big_count
    print(col, mean)

size 260.55856666666665
age 24.534966666666666
rate1 0.5004729000000001
rate2 1.0012718532009215
